# SARIMAX Training Pipeline
## Semester-Aware Forecasting with Fixed Parameters (2,1,1)x(0,0,1,6/12)

This notebook trains and saves a SARIMAX model with domain knowledge about the Philippine university calendar.

**Key Features:**
- Fixed parameters: (2,1,1) nonseasonal × (0,0,1) with s=6 (or 12 when 36+ months available)
- Semester-aware exogenous features (exam weeks, semester start, working days)
- Automatic fallback to CSV if database is empty
- Model artifact saved for use in Flask backend

## 1. Setup and Configuration

In [1]:
import os
import pickle
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import calendar
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Configuration
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
ARTIFACT_DIR = os.path.join(BASE_DIR, 'model_artifacts')
ARTIFACT_PATH = os.path.join(ARTIFACT_DIR, 'sarimax_model.pkl')
MONTHLY_CSV_FALLBACK = os.path.join(BASE_DIR, 'data_mining', 'EMC_Monthly_Reservations_Cleaned.csv')

print(f"Base directory: {BASE_DIR}")
print(f"Artifact path: {ARTIFACT_PATH}")
print(f"CSV fallback: {MONTHLY_CSV_FALLBACK}")

## 2. Feature Engineering: Semester-Aware Calendar Features

In [ ]:
def build_features(index):
    """
    Build semester-aware calendar features for each month.

    Semester calendar (Philippine university):
      1st Semester : Aug – Dec  (5 months)
      2nd Semester : Jan – May  (5 months)
      Summer       : Jun – Jul  (2 months)

    Exam schedule: every 5 weeks (1st and 2nd semester ONLY)
      Month 2 of 1st/2nd sem   = midterm exam month
      Month 4-5 of 1st/2nd sem = finals exam months
      Summer semester (Jun-Jul) has NO exams
    """
    rows = []
    for date in index:
        m, y = date.month, date.year

        if m in [8, 9, 10, 11, 12]:
            sem_num = 1
            month_in_sem = m - 7       # Aug=1, Sep=2, Oct=3, Nov=4, Dec=5
        elif m in [1, 2, 3, 4, 5]:
            sem_num = 2
            month_in_sem = m           # Jan=1, Feb=2, Mar=3, Apr=4, May=5
        else:
            sem_num = 3
            month_in_sem = m - 5       # Jun=1, Jul=2

        # No exams during summer semester
        is_exam   = 1 if (sem_num in [1, 2] and month_in_sem == 2) else 0
        is_finals = 1 if (sem_num in [1, 2] and month_in_sem >= 4) else 0
        is_start  = 1 if month_in_sem == 1 else 0
        is_summer = 1 if sem_num == 3 else 0

        _, dim = calendar.monthrange(y, m)
        wdays  = sum(1 for d in range(1, dim + 1)
                     if pd.Timestamp(y, m, d).weekday() < 5)

        rows.append({
            'month_in_sem' : month_in_sem,
            'is_exam'      : is_exam,
            'is_finals'    : is_finals,
            'is_sem_start' : is_start,
            'is_summer'    : is_summer,
            'working_days' : wdays,
        })
    return pd.DataFrame(rows, index=index)

print("Feature builder function defined.")

## 3. Data Loading Functions

In [ ]:
def _load_monthly_series_from_csv(csv_path=MONTHLY_CSV_FALLBACK):
    """Load reservation data from CSV fallback file."""
    if not os.path.exists(csv_path):
        raise ValueError('No reservation data in DB and fallback monthly CSV not found.')

    df = pd.read_csv(csv_path)
    if 'month' not in df.columns or 'reservation_count' not in df.columns:
        raise ValueError('Fallback monthly CSV must contain month and reservation_count columns.')

    df['month'] = pd.to_datetime(df['month'])
    df = df.sort_values('month')
    series = pd.Series(df['reservation_count'].astype(float).values, index=df['month'])
    series = series.asfreq('MS', fill_value=0.0)
    series.name = 'reservation_count'
    return series

print("CSV loader function defined.")

In [ ]:
def build_monthly_reservation_series(include_statuses=None):
    """Aggregate reservation counts into a monthly time series from database."""
    # NOTE: This function requires Flask app context to access database
    try:
        from models import Reservation
        
        query = Reservation.query
        if include_statuses:
            query = query.filter(Reservation.status.in_(include_statuses))
        query = query.filter(Reservation.start_time <= datetime.utcnow())

        rows = query.with_entities(Reservation.start_time).all()
        timestamps = [row[0] for row in rows if row[0] is not None]
        if not timestamps:
            print("No reservations in database, using CSV fallback...")
            return _load_monthly_series_from_csv()

        series = pd.Series(1, index=pd.to_datetime(timestamps), dtype='int64')
        monthly = series.resample('MS').sum().astype(float)
        monthly = monthly.asfreq('MS', fill_value=0.0)
        monthly.name = 'reservation_count'
        return monthly
    except ImportError:
        print("Database not available, using CSV fallback...")
        return _load_monthly_series_from_csv()

print("Data builder functions defined.")

## 4. SARIMAX Parameter Selection

In [ ]:
def get_sarimax_parameters(series):
    """
    Determine SARIMAX parameters based on data length.
    
    - With < 36 months: use (2,1,1)x(0,0,1,6)  — 6-month seasonality (semester cycle)
    - With >= 36 months: use (2,1,1)x(0,0,1,12) — 12-month seasonality (annual cycle)
    
    This allows the model to capture annual patterns once enough data is available.
    """
    order = (2, 1, 1)
    
    if len(series) >= 36:
        # 36+ months available: switch to 12-month seasonality for annual patterns
        seasonal_order = (0, 0, 1, 12)
        seasonality = "12-month (annual)"
    else:
        # < 36 months: use 6-month seasonality (semester cycle)
        seasonal_order = (0, 0, 1, 6)
        seasonality = "6-month (semester)"
    
    print(f"Data length: {len(series)} months")
    print(f"Selected seasonality: {seasonality}")
    print(f"SARIMAX order: {order}x{seasonal_order}")
    
    return order, seasonal_order

print("Parameter selection function defined.")

## 5. Model Training

In [ ]:
def train_sarimax_model(series):
    """Train SARIMAX with fixed parameters (2,1,1)x(0,0,1,6/12) on full series."""
    print("Step 1: Determining parameters...")
    order, seasonal_order = get_sarimax_parameters(series)
    
    print("\nStep 2: Building exogenous features...")
    exog = build_features(series.index)
    print(f"  Features created for {len(exog)} months")
    print(f"  Feature columns: {list(exog.columns)}")
    
    print("\nStep 3: Training SARIMAX model...")
    model = SARIMAX(
        series,
        exog=exog,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    print(f"  Model trained successfully")
    print(f"  AIC: {model.aic:.2f}")
    print(f"  BIC: {model.bic:.2f}")

    return model, order, seasonal_order

print("Training function defined.")

## 6. Model Persistence

In [ ]:
def save_sarimax_bundle(model, series, order, seasonal_order, artifact_path=ARTIFACT_PATH):
    """Save SARIMAX model and metadata to pickle file."""
    os.makedirs(os.path.dirname(artifact_path), exist_ok=True)

    bundle = {
        'model': model,
        'metadata': {
            'model_type': 'SARIMAX with Semester Features',
            'order': order,
            'seasonal_order': seasonal_order,
            'trained_at': datetime.utcnow().isoformat() + 'Z',
            'n_observations': int(len(series)),
            'last_observation_month': series.index[-1].strftime('%Y-%m'),
        },
    }

    with open(artifact_path, 'wb') as f:
        pickle.dump(bundle, f)

    print(f"\nModel saved to: {artifact_path}")
    print(f"Metadata: {bundle['metadata']}")
    
    return bundle['metadata']

print("Save function defined.")

## 7. Main Retraining Orchestration

In [ ]:
def retrain_all_historical_data(include_statuses=None, artifact_path=ARTIFACT_PATH):
    """Train SARIMAX using all historical data and persist model artifact."""
    print("="*70)
    print("SARIMAX RETRAINING PIPELINE")
    print("="*70)
    
    print("\nPhase 1: Loading reservation data...")
    series = build_monthly_reservation_series(include_statuses=include_statuses)
    print(f"  Loaded {len(series)} months")
    print(f"  Date range: {series.index[0].strftime('%b %Y')} to {series.index[-1].strftime('%b %Y')}")
    print(f"  Min reservations: {series.min():.0f}")
    print(f"  Max reservations: {series.max():.0f}")
    print(f"  Mean reservations: {series.mean():.1f}")
    
    print("\nPhase 2: Training model...")
    model, order, seasonal_order = train_sarimax_model(series)
    
    print("\nPhase 3: Saving model artifact...")
    metadata = save_sarimax_bundle(model, series, order, seasonal_order, artifact_path=artifact_path)
    
    print("\n" + "="*70)
    print("RETRAINING COMPLETED SUCCESSFULLY")
    print("="*70)
    
    return metadata

print("Main orchestration function defined.")

## 8. Execute Retraining

**Note:** This cell requires Flask app context. If running standalone, uncomment the app context code below.

In [ ]:
# Uncomment the following lines if running this notebook standalone (outside Flask)
# This will use the CSV fallback data

# Execute retraining with approved reservations only
result = retrain_all_historical_data(include_statuses=['approved'])

print("\n" + "="*70)
print("METADATA SUMMARY")
print("="*70)
for key, value in result.items():
    print(f"{key:<25}: {value}")

## 9. Alternative: Run within Flask App Context

Use this cell when running from the Flask application server to access the database.

In [ ]:
# Uncomment to run with database access from Flask app
#
# from app import app
# 
# with app.app_context():
#     result = retrain_all_historical_data(include_statuses=['approved'])
#     print("\nSARIMAX Retraining completed successfully!")
#     for key, value in result.items():
#         print(f"  {key}: {value}")